# TabPFN timing / memory benchmark  (without_HPO | with_HPO | RF+TabPFN)

Reproduces each variant's construction from the `*_Calc` notebooks and measures, per
**(variant x target)**:

| metric | how |
|---|---|
| Training wall time [s] | `perf_counter` around `.fit` |
| Training GPU time [s]  | `cuda.Event` elapsed across `.fit` |
| Training CPU time [s]  | `process_time` around `.fit` |
| Per-sample inference [ms] | `predict` on the test split / n_test |
| Used RAM [MB] | peak process RSS rise during `.fit` |
| GPU memory [MB] | `torch.cuda.max_memory_allocated` |
| Model size [MB] | `joblib.dump` size |

Runs **in-process**, one combo at a time, freeing each model (`del`+`gc`+`empty_cache`)
before the next so RAM/GPU-mem stay per-combo. Run on an **H100** with the `py_a6` kernel.
With `N_TRIALS=100`, the `with_HPO` and `RF+TabPFN` rows are the slow part (hours total).

In [ ]:
import os, gc, time, tempfile, threading
import numpy as np, pandas as pd

# thread env (fidelity with the SLURM runs) — set BEFORE importing torch/tabpfn
n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = str(n_cpus)
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

import torch, joblib, psutil
from sklearn.model_selection import train_test_split
try: torch.set_num_threads(n_cpus)
except Exception: pass
CUDA = torch.cuda.is_available()
print("CUDA:", CUDA, (torch.cuda.get_device_name(0) if CUDA else ""))

In [ ]:
N_TRIALS = 100          # with_HPO TPE trials (matches the runs)
TARGETS  = ["P_bubble", "P_dew", "gamma", "interfacial_thickness"]
VARIANTS = ["without_HPO", "with_HPO", "RF_TabPFN"]
OUTDIR   = "BENCHMARK_TIMING"

# per-(variant, target) seed, copied from each *_Calc notebook
SEEDS = {
    "without_HPO": {"P_bubble": 50015,  "P_dew": 855015,  "gamma": 50005,  "interfacial_thickness": 655552},
    "with_HPO":    {"P_bubble": 454015, "P_dew": 6702315, "gamma": 844015, "interfacial_thickness": 655552},
    "RF_TabPFN":   {"P_bubble": 50015,  "P_dew": 855015,  "gamma": 50005,  "interfacial_thickness": 655552},
}
DATA_CANDIDATES = [
    os.environ.get("TABPFN_DATASET", ""),
    "/gpfs/home6/draju/A6/DATASET_A4/interfacial_results_dataset_A4.csv",
    "/home/darshan/A6/PCSAFT_cDFT/PART_1/interfacial_results_dataset_A4.csv",
]

In [ ]:
def dataset_path():
    for p in DATA_CANDIDATES:
        if p and os.path.exists(p):
            return p
    raise FileNotFoundError(f"dataset not found in {DATA_CANDIDATES}")

def load_split(target, seed):
    df = pd.read_csv(dataset_path())
    z = [c for c in df.columns if c.startswith("z_") and (df[c] != 0).any()]
    features = ["temperature", "pressure"] + z
    X, y = df[features], df[target]
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, random_state=seed)
    X_te, X_va, y_te, y_va   = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=seed)
    return X_tr, y_tr, X_te, y_te

def build_model(variant, seed):
    from tabpfn import TabPFNRegressor
    if variant == "without_HPO":
        return TabPFNRegressor(random_state=seed, ignore_pretraining_limits=True,
                               fit_mode="fit_preprocessors")
    if variant == "RF_TabPFN":
        from tabpfn_extensions.rf_pfn import RandomForestTabPFNRegressor
        base = TabPFNRegressor(random_state=seed, ignore_pretraining_limits=True,
                               fit_mode="fit_preprocessors")
        return RandomForestTabPFNRegressor(tabpfn=base, n_estimators=10, max_depth=3)
    if variant == "with_HPO":
        from tabpfn_extensions.hpo import TunedTabPFNRegressor
        from tabpfn_extensions.hpo.search_space import get_param_grid_hyperopt
        from tabpfn.constants import ModelVersion
        ss = get_param_grid_hyperopt("regression", model_version=ModelVersion.V2_5)
        ss["ignore_pretraining_limits"] = True
        return TunedTabPFNRegressor(n_trials=N_TRIALS, metric="rmse", n_validation_size=0.2,
                                    shuffle_data=True, search_algorithm_type="tpe", device="auto",
                                    random_state=seed, verbose=False, search_space=ss)
    raise ValueError(variant)

class RSSPeak:
    # peak process RSS (MB) rise over the with-block
    def __init__(self, dt=0.05):
        self.p = psutil.Process(); self.dt = dt
        self.base = self.p.memory_info().rss; self.peak = self.base
    def __enter__(self):
        self._run = True
        self.t = threading.Thread(target=self._loop, daemon=True); self.t.start(); return self
    def _loop(self):
        while self._run:
            self.peak = max(self.peak, self.p.memory_info().rss); time.sleep(self.dt)
    def __exit__(self, *a):
        self._run = False; self.t.join(timeout=1.0)
    @property
    def peak_mb(self): return (self.peak - self.base) / 1e6

In [ ]:
def run_one(variant, target):
    seed = SEEDS[variant][target]
    X_tr, y_tr, X_te, y_te = load_split(target, seed)
    model = build_model(variant, seed)

    if CUDA:
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        ev0, ev1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)

    with RSSPeak() as rss:                       # ---- training ----
        if CUDA: ev0.record()
        tw0, tc0 = time.perf_counter(), time.process_time()
        model.fit(X_tr, y_tr)
        if CUDA: torch.cuda.synchronize()
        train_wall, train_cpu = time.perf_counter() - tw0, time.process_time() - tc0
        if CUDA: ev1.record(); torch.cuda.synchronize()
    train_gpu = ev0.elapsed_time(ev1) / 1e3 if CUDA else float("nan")
    gpu_mem   = torch.cuda.max_memory_allocated() / 1e6 if CUDA else float("nan")

    t0 = time.perf_counter()                     # ---- inference ----
    model.predict(X_te)
    if CUDA: torch.cuda.synchronize()
    ips = (time.perf_counter() - t0) / len(X_te)

    with tempfile.NamedTemporaryFile(suffix=".joblib") as f:   # ---- size ----
        joblib.dump(model, f.name); size_mb = os.path.getsize(f.name) / 1e6

    row = dict(variant=variant, target=target, seed=seed,
               n_train=len(X_tr), n_test=len(X_te),
               train_wall_s=round(train_wall, 4), train_gpu_s=round(train_gpu, 4),
               train_cpu_s=round(train_cpu, 4), infer_per_sample_ms=ips * 1e3,
               used_ram_mb=round(rss.peak_mb, 4), gpu_mem_mb=round(gpu_mem, 4),
               model_size_mb=round(size_mb, 4))
    del model; gc.collect()
    if CUDA: torch.cuda.empty_cache()
    return row

In [ ]:
rows = []
for variant in VARIANTS:
    for target in TARGETS:
        print(f"=== {variant} / {target} ===", flush=True)
        try:
            r = run_one(variant, target); rows.append(r)
            print(f"  wall={r['train_wall_s']}s gpu={r['train_gpu_s']}s "
                  f"infer={r['infer_per_sample_ms']:.4f}ms ram={r['used_ram_mb']}MB "
                  f"gpumem={r['gpu_mem_mb']}MB size={r['model_size_mb']}MB", flush=True)
        except Exception:
            import traceback; traceback.print_exc()

os.makedirs(OUTDIR, exist_ok=True)
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUTDIR, "benchmark_timing.csv"), index=False)
df

In [ ]:
# ---- Table-4-style tables: rows = metrics, columns = variants (one per target) ----
ROWS = [("Training wall time [s]",      "train_wall_s"),
        ("Training GPU time [s]",       "train_gpu_s"),
        ("Per-sample inference [ms]",   "infer_per_sample_ms"),
        ("Used RAM [MB]",               "used_ram_mb"),
        ("GPU memory [MB]",             "gpu_mem_mb"),
        ("Model size [MB]",             "model_size_mb")]

def table_for(target):
    sub = df[df.target == target].set_index("variant")
    data = {lab: [sub.loc[v, key] if v in sub.index else np.nan for v in VARIANTS]
            for lab, key in ROWS}
    return pd.DataFrame(data, index=VARIANTS).T

with open(os.path.join(OUTDIR, "timing_tables.tex"), "w") as f:
    for tgt in TARGETS:
        t = table_for(tgt)
        print(f"\n### {tgt}"); display(t)
        f.write(t.to_latex(float_format="%.4f", caption=f"TabPFN timing/memory --- {tgt}",
                           label=f"tab:timing_{tgt}"))
        f.write("\n")
print("wrote", os.path.join(OUTDIR, "timing_tables.tex"))